# AiStock V10 测试 Notebook

## 配置-功能解耦架构全面测试

V10 核心架构升级:
- **配置-功能解耦**: 所有硬编码配置 → YAML 独立文件
- **统一配置加载服务**: ConfigService (多系统隔离 + 热重载 + 4层合并)
- **统一服务层**: CacheService / DataLoaderService / LoggerService (跨子系统共用)
- **事件总线**: EventBus (子系统间数据交互)
- **服务容器**: ServiceContainer (依赖注入)
- **子系统架构**: market_state / price_quant (独立隔离)

测试分为4大模块:
1. **基础设施层测试** — ConfigService / CacheService / EventBus / ServiceContainer
2. **数据服务层测试** — TDXAdapter / AKAdapter / DatabaseReader / DataLoaderService
3. **市场状态子系统测试** — ContractManager / OptionCodeParser / DerivativesSignalEngine
4. **端到端管线测试** — bootstrap → run_pipeline 完整流程

In [1]:
import sys, os
from pathlib import Path

# 确保项目根目录在 sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python: {sys.version}')

# 检查关键目录
for d in ['base_service', 'config/yaml', 'data_service', 'subsystems/market_state', 'data']:
    p = PROJECT_ROOT / d
    print(f'  {d}: {"EXISTS" if p.exists() else "MISSING"} ({len(list(p.glob("*")))} files)' if p.exists() else f'  {d}: MISSING')

PROJECT_ROOT: /home/ts/app/AiStock_v10
Python: 3.14.3 (main, Feb  6 2026, 23:41:13) [GCC 13.3.0]
  base_service: EXISTS (7 files)
  config/yaml: EXISTS (8 files)
  data_service: EXISTS (6 files)
  subsystems/market_state: EXISTS (5 files)
  data: EXISTS (2 files)


---
## 模块1: 基础设施层测试

### 1.1 ConfigService — 统一配置加载服务

In [2]:
# ============================================================
# 1.1 ConfigService — 多文件YAML加载 + 点号路径访问
# ============================================================
from base_service.config_service import ConfigService

config = ConfigService(
    config_dir=str(PROJECT_ROOT / "config" / "yaml"),
    enable_hot_reload=False,  # 测试时关闭热重载
)
config.load_all()

print(f'已加载 YAML 文件数: {len(config._configs)}')
print(f'已加载的文件列表: {list(config._configs.keys())}')
print()

# ---- 测试: 点号路径访问 ----
print('=== 点号路径访问测试 ===')
version = config.get('system.version', 'N/A')
mode = config.get('system.mode', 'N/A')
std_host = config.get('tdx.standard.host', 'N/A')
std_port = config.get('tdx.standard.port', 0, value_type=int)
ext_host = config.get('tdx.extension.host', 'N/A')
ext_port = config.get('tdx.extension.port', 0, value_type=int)

print(f'  system.version:        {version}')
print(f'  system.mode:           {mode}')
print(f'  tdx.standard.host:     {std_host}')
print(f'  tdx.standard.port:     {std_port}')
print(f'  tdx.extension.host:    {ext_host}')
print(f'  tdx.extension.port:    {ext_port}')
print()

# ---- 测试: 配置节获取 ----
print('=== 配置节获取测试 ===')
cache_cfg = config.get_section('cache')
print(f'  cache section: {cache_cfg}')
print()

# ---- 测试: codes.yaml 便捷方法 ----
print('=== codes.yaml 便捷方法测试 ===')
futures = config.get_futures_codes()
indices = config.get_index_codes()
underlyings = config.get_option_underlyings()

print(f'  期货代码数: {len(futures)}')
for f in futures[:5]:
    print(f'    {f.get("code", "?")} ({f.get("name", "?")}) — market_type={f.get("market_type", "?")}')
if len(futures) > 5:
    print(f'    ... 共 {len(futures)} 个')
print()

print(f'  指数代码数: {len(indices)}')
for idx in indices:
    print(f'    {idx.get("code", "?")} ({idx.get("name", "?")})')
print()

print(f'  期权标的数: {len(underlyings)}')
for key, info in list(underlyings.items())[:5]:
    print(f'    {key} ({info.get("name", "?")}) — market={info.get("market_code", "?")}')
if len(underlyings) > 5:
    print(f'    ... 共 {len(underlyings)} 个')
print()

# ---- 测试: 品种-市场映射 ----
print('=== 品种-市场映射测试 ===')
variety_market = config.get_variety_market()
print(f'  品种数: {len(variety_market)}')
for v in list(variety_market.keys())[:5]:
    info = variety_market[v]
    print(f'    {v}: market_code={info.get("market_code")}, market_type={info.get("market_type")}, name={info.get("name")}')
print()

# ---- 测试: 交割月映射 ----
print('=== 交割月映射测试 ===')
delivery_months = config.get_delivery_months()
print(f'  品种数: {len(delivery_months)}')
for v in list(delivery_months.keys())[:5]:
    print(f'    {v}: {delivery_months[v]}')
print()

# ---- 测试: 子系统配置4层合并 ----
print('=== 子系统配置4层合并测试 ===')
ms_config = config.get_subsystem_config('market_state')
print(f'  market_state 子系统配置键数: {len(ms_config)}')
print(f'  commodity_signal_weights: {ms_config.get("commodity_signal_weights", {})}')
print(f'  composite_weights: {ms_config.get("composite_weights", {})}')

print('\nConfigService 所有测试通过!')

已加载 YAML 文件数: 8
已加载的文件列表: ['cache', 'codes', 'database', 'logging', 'market_state', 'price_quant', 'system', 'tdx']

=== 点号路径访问测试 ===
  system.version:        10.0
  system.mode:           production
  tdx.standard.host:     180.153.18.170
  tdx.standard.port:     7709
  tdx.extension.host:    180.153.18.176
  tdx.extension.port:    7721

=== 配置节获取测试 ===
  cache section: {'backend': 'memory', 'default_ttl': 300, 'max_size': 1000, 'namespaces': {'config': {'ttl': 0, 'max_size': 100}, 'data': {'ttl': 300, 'max_size': 500}, 'computation': {'ttl': 600, 'max_size': 200}}}

=== codes.yaml 便捷方法测试 ===
  期货代码数: 19
    IFL8 (沪深300主连) — market_type=future_zj
    ICL8 (中证500主连) — market_type=future_zj
    IML8 (中证1000主连) — market_type=future_zj
    IHL8 (上证50主连) — market_type=future_zj
    TFL8 (5年期国债主连) — market_type=future_zj
    ... 共 19 个

  指数代码数: 6
    000001 (上证指数)
    399001 (深证成指)
    399006 (创业板指)
    000300 (沪深300)
    000905 (中证500)
    000852 (中证1000)

  期权标的数: 52
    IO (沪深300指数期权) —

### 1.2 CacheService — 命名空间隔离缓存

In [3]:
# ============================================================
# 1.2 CacheService — 命名空间隔离 + TTL
# ============================================================
from base_service.cache_service import CacheService

cache = CacheService(default_ttl=60, default_max_size=100)

# 测试基本操作
cache.set('test_key', {'value': 42}, ttl=30)
result = cache.get('test_key')
print(f'基本 get/set: {result}')

# 测试命名空间
ns_data = cache.namespace('data')
ns_data.set('000300', {'name': '沪深300', 'close': 3800.5})
ns_data.set('000905', {'name': '中证500', 'close': 5200.3})

ns_comp = cache.namespace('computation')
ns_comp.set('signal_000300', 65.2)

print(f'命名空间 data.get("000300"): {ns_data.get("000300")}')
print(f'命名空间 computation.get("signal_000300"): {ns_comp.get("signal_000300")}')

# 测试统计
print(f'缓存统计: {cache.stats}')

# 测试批量操作
ns_data.set_batch({'000852': {'name': '中证1000'}, '399006': {'name': '创业板指'}})
batch_result = ns_data.get_batch(['000300', '000905', '000852', '399006'])
print(f'批量获取: {list(batch_result.keys())}')

print('CacheService 测试通过!')

基本 get/set: {'value': 42}
命名空间 data.get("000300"): {'name': '沪深300', 'close': 3800.5}
命名空间 computation.get("signal_000300"): 65.2
缓存统计: {'config': {'size': 0, 'max_size': 100, 'hits': 0, 'misses': 0, 'hit_rate': 0.0, 'evictions': 0}, 'data': {'size': 3, 'max_size': 500, 'hits': 2, 'misses': 0, 'hit_rate': 1.0, 'evictions': 0}, 'computation': {'size': 1, 'max_size': 200, 'hits': 1, 'misses': 0, 'hit_rate': 1.0, 'evictions': 0}, 'contract': {'size': 0, 'max_size': 100, 'hits': 0, 'misses': 0, 'hit_rate': 0.0, 'evictions': 0}}
批量获取: ['000300', '000905', '000852', '399006']
CacheService 测试通过!


### 1.3 EventBus — 事件总线

In [4]:
# ============================================================
# 1.3 EventBus — Topic发布/订阅 + 通配符
# ============================================================
from base_service.event_bus import EventBus, Event, Topics

bus = EventBus(history_size=100)

# 测试基本发布/订阅
received = []
def handler(event: Event):
    received.append(event)
    print(f'  收到事件: topic={event.topic}, data={event.data}, source={event.source}')

bus.subscribe('config.changed', handler)
bus.publish('config.changed', {'key': 'tdx.standard.port', 'old': 7709, 'new': 7709}, source='test')

print(f'收到事件数: {len(received)}')

# 测试通配符
wildcard_received = []
def wildcard_handler(event: Event):
    wildcard_received.append(event)

bus.subscribe('market_state.*', wildcard_handler)
bus.publish('market_state.updated', {'signal': 65.2}, source='engine')
bus.publish('market_state.warning', {'warnings': 2}, source='engine')

print(f'通配符匹配事件数: {len(wildcard_received)}')

# 测试历史回放
history = bus.get_history('market_state.*', limit=5)
print(f'历史回放: {len(history)} 条')

# 测试预定义Topics
print(f'预定义 Topics: {Topics.CONFIG_CHANGED}, {Topics.DATA_LOADED}, {Topics.MARKET_STATE_UPDATED}')

print('EventBus 测试通过!')

  收到事件: topic=config.changed, data={'key': 'tdx.standard.port', 'old': 7709, 'new': 7709}, source=test
收到事件数: 1
通配符匹配事件数: 2
历史回放: 2 条
预定义 Topics: config.changed, data.loaded, market_state.updated
EventBus 测试通过!


### 1.4 ServiceContainer — 服务容器 + SubsystemBase

In [5]:
# ============================================================
# 1.4 ServiceContainer — DI容器 + SubsystemBase
# ============================================================
from base_service.service_container import ServiceContainer, SubsystemBase

container = ServiceContainer()

# 注册单例
container.register_singleton('config', config)
container.register_singleton('cache', cache)
container.register_singleton('event_bus', bus)

# 注册工厂
container.register_factory('my_service', lambda: {'created_at': 'now', 'type': 'factory'})

# 测试获取
print(f'获取 config: {type(container.get("config")).__name__}')
print(f'获取 cache: {type(container.get("cache")).__name__}')
print(f'获取工厂 my_service: {container.get("my_service")}')
print(f'已注册服务: {container.list_services()}')
print(f'has("config"): {container.has("config")}')
print(f'has("unknown"): {container.has("unknown")}')

# 测试 SubsystemBase
class TestSubsystem(SubsystemBase):
    def __init__(self, container):
        super().__init__('test_subsystem', container)
    
    def run(self):
        return {'status': 'ok', 'name': self._name}

test_sub = TestSubsystem(container)
print(f'子系统名称: {test_sub.name}')
print(f'子系统配置: {type(test_sub.config).__name__}')
print(f'子系统运行: {test_sub.run()}')

print('ServiceContainer + SubsystemBase 测试通过!')

获取 config: ConfigService
获取 cache: CacheService
获取工厂 my_service: {'created_at': 'now', 'type': 'factory'}
已注册服务: ['event_bus', 'my_service', 'config', 'cache']
has("config"): True
has("unknown"): False
子系统名称: test_subsystem
子系统配置: ConfigService
子系统运行: {'status': 'ok', 'name': 'test_subsystem'}
ServiceContainer + SubsystemBase 测试通过!


---
## 模块2: 数据服务层测试

### 2.1 TDXAdapter — 通达信双端口适配器

In [6]:
# ============================================================
# 2.1 TDXAdapter — 双端口 (7709/7721) 配置驱动
# ============================================================
from data_service.tdx_adapter import TDXAdapter

# 使用 ConfigService 初始化
tdx = TDXAdapter(config_service=config)

print(f'TDX 标准端口: {tdx._std_host}:{tdx._std_port}')
print(f'TDX 扩展端口: {tdx._ext_host}:{tdx._ext_port}')
print(f'连接池大小: {tdx._std_pool.max_conn}')
print()

# ---- 健康检查 ----
print('=== TDX 健康检查 ===')
health = tdx.health_check()
print(f'  标准端口: {health.get("standard_port", False)}')
print(f'  扩展端口: {health.get("extension_port", False)}')
print()

# ---- 获取指数K线 (标准端口) ----
if health.get('standard_port', False):
    print('=== 指数K线测试 (标准端口) ===')
    df = tdx.get_index_daily(code='000300', count=10)
    if not df.empty:
        print(f'  沪深300 最新 {len(df)} 条数据')
        print(f'  列: {list(df.columns)}')
        print(f'  最新收盘价: {df.iloc[-1].get("close", "N/A")}')
    else:
        print('  数据为空')
    print()

    # ---- 获取股票K线 ----
    print('=== 股票K线测试 (标准端口) ===')
    df_stock = tdx.get_stock_daily(code='600000', count=5)
    if not df_stock.empty:
        print(f'  浦发银行 最新 {len(df_stock)} 条数据')
    else:
        print('  股票数据为空')
    print()

# ---- 获取期货K线 (扩展端口) ----
if health.get('extension_port', False):
    print('=== 期货K线测试 (扩展端口) ===')
    # 从 ConfigService 获取期货代码
    for fc in futures[:3]:
        code = fc.get('code', '')
        name = fc.get('name', '')
        mt = fc.get('market_type', '')
        df_f = tdx.get_future_daily(code=code, market_type=mt, count=5)
        if not df_f.empty:
            print(f'  {name}({code}): {len(df_f)}条, 最新close={df_f.iloc[-1].get("close", "N/A")}')
        else:
            print(f'  {name}({code}): 数据为空')
    print()

    # ---- 合约列表 ----
    print('=== 合约列表测试 (扩展端口) ===')
    df_inst = tdx.get_instrument_info(market=47)
    print(f'  中金期货合约数: {len(df_inst)}')
    if not df_inst.empty:
        print(f'  示例: {df_inst["code"].head(3).tolist() if "code" in df_inst.columns else "无code列"}')

print('\nTDXAdapter 测试完成!')

TDX 标准端口: 180.153.18.170:7709
TDX 扩展端口: 180.153.18.176:7721
连接池大小: 3

=== TDX 健康检查 ===
  标准端口: True
  扩展端口: True

=== 指数K线测试 (标准端口) ===
  沪深300 最新 10 条数据
  列: ['open', 'close', 'high', 'low', 'volume', 'amount', 'year', 'month', 'day', 'hour', 'minute', 'datetime', 'up_count', 'down_count', 'date']
  最新收盘价: 4947.85

=== 股票K线测试 (标准端口) ===
  浦发银行 最新 5 条数据

=== 期货K线测试 (扩展端口) ===
  沪深300主连(IFL8): 5条, 最新close=4899.2001953125
  中证500主连(ICL8): 5条, 最新close=8565.0009765625
  中证1000主连(IML8): 5条, 最新close=8596.0009765625

=== 合约列表测试 (扩展端口) ===
  中金期货合约数: 69
  示例: ['IC2606', 'IC2607', 'IC2609']

TDXAdapter 测试完成!


### 2.2 AKAdapter — 海外期货适配器

In [7]:
# ============================================================
# 2.2 AKAdapter — 海外期货 + 辅助数据
# ============================================================
from data_service.ak_adapter import AKAdapter

ak = AKAdapter(config_service=config)
print(f'AKAdapter 初始化完成')

# 健康检查
ak_health = ak.health_check()
print(f'AKShare 健康: {ak_health}')
print()

# ---- 海外期货数据 ----
print('=== 海外期货Core层测试 ===')
core_data = ak.get_futures_by_tier('base_service')
print(f'  Core层品种数: {len(core_data)}')
for sym, data in list(core_data.items())[:3]:
    print(f'    {sym} ({data.get("name", "?")}): price={data.get("price", "N/A")}')
print()

# ---- 辅助数据 ----
print('=== 辅助数据测试 (QVIX 50ETF) ===')
qvix = ak.get_auxiliary_data('qvix_50etf')
if qvix is not None and not qvix.empty:
    print(f'  QVIX 50ETF 数据行数: {len(qvix)}')
    print(f'  列: {list(qvix.columns)[:5]}')
else:
    print('  QVIX 数据不可用 (可能需要akshare版本支持)')

print('\nAKAdapter 测试完成!')

未知层级: base_service


AKAdapter 初始化完成
AKShare 健康: {'akshare': True, 'version': '1.18.63', 'symbols_loaded': 29}

=== 海外期货Core层测试 ===
  Core层品种数: 0

=== 辅助数据测试 (QVIX 50ETF) ===


QVIX 50ETF接口不可用


  QVIX 数据不可用 (可能需要akshare版本支持)

AKAdapter 测试完成!


### 2.3 DatabaseReader — PostgreSQL估值数据

In [8]:
# ============================================================
# 2.3 DatabaseReader — PostgreSQL (10.3.18.56)
# ============================================================
from data_service.database_reader import DatabaseReader

db = DatabaseReader(config_service=config)

print(f'DatabaseReader 连接状态: {db.is_connected}')
if db.is_connected:
    # 测试PE数据
    pe = db.get_index_pe('000300', days=5)
    print(f'沪深300 PE数据: {len(pe)} 行')
    if not pe.empty:
        print(pe.tail(3))
else:
    print('数据库未连接 (需配置 10.3.18.56 的PostgreSQL)')

print('\nDatabaseReader 测试完成!')

DatabaseReader 连接失败: No module named 'psycopg2'


DatabaseReader 连接状态: False
数据库未连接 (需配置 10.3.18.56 的PostgreSQL)

DatabaseReader 测试完成!


### 2.4 DataLoaderService — 7段数据编排

In [9]:
# ============================================================
# 2.4 DataLoaderService — 统一数据加载编排
# ============================================================
from data_service.data_loader_service import DataLoaderService

data_loader = DataLoaderService(
    tdx_adapter=tdx,
    ak_adapter=ak,
    db_reader=db,
    config_service=config,
)

print(f'DataLoaderService 初始化完成')
print(f'  索引代码数: {len(config.get_index_codes())}')
print(f'  期货代码数: {len(config.get_futures_codes())}')
print(f'  期权标的数: {len(config.get_option_underlyings())}')
print()

# ---- 按段加载测试 ----
print('=== 按段加载: INDEX_DATA ===')
index_data = data_loader.load_section('index_data')
if isinstance(index_data, dict):
    for code, df in index_data.items():
        rows = len(df) if hasattr(df, '__len__') else 0
        print(f'  {code}: {rows} rows')
print()

print('=== 按段加载: FUTURES_DATA ===')
futures_data = data_loader.load_section('futures_data')
if isinstance(futures_data, dict):
    for code, df in futures_data.items():
        rows = len(df) if hasattr(df, '__len__') else 0
        print(f'  {code}: {rows} rows')
print()

# ---- 质量报告 ----
quality = data_loader.get_quality_report()
print('=== 数据质量报告 ===')
for section, report in quality.items():
    print(f'  {section}: success_rate={report.get("success_rate", "?")}, healthy={report.get("is_healthy", "?")}')

print('\nDataLoaderService 测试完成!')

DataLoaderService 初始化完成
  索引代码数: 6
  期货代码数: 19
  期权标的数: 52

=== 按段加载: INDEX_DATA ===


ValueError: Unable to coerce to Series, length must be 15: given 0

---
## 模块3: 市场状态子系统测试

### 3.1 ContractManager — 动态合约推导 (全配置驱动)

In [ ]:
# ============================================================
# 3.1 ContractManager — V10 全配置驱动
# ============================================================
from subsystems.market_state.core.contract_manager import ContractManager

# 检查代码表文件
code_table_path = str(PROJECT_ROOT / 'data' / 'tdxAPICode180.xlsx')
print(f'代码表路径: {code_table_path}')
print(f'代码表存在: {os.path.exists(code_table_path)}')

cm = ContractManager(
    config=config,
    code_table_path=code_table_path if os.path.exists(code_table_path) else None,
)
print(f'ContractManager 初始化完成')
print(f'  合约数: {len(cm.contracts)}')
print()

# ---- 商品期货合约推导 ----
print('=== 商品期货合约推导测试 ===')
test_varieties = ['CU', 'AL', 'AU', 'AG', 'RB']
for variety in test_varieties:
    pair = cm.get_commodity_contracts(variety_code=variety)
    if pair:
        print(f'  {variety}: near={pair.near_month.code if pair.near_month else "?"}, '
              f'far={pair.far_month.code if pair.far_month else "?"}')
    else:
        print(f'  {variety}: 无数据')
print()

# ---- 股指期货合约推导 ----
print('=== 股指期货合约推导测试 ===')
for key in ['if', 'ih', 'ic', 'im']:
    contract = cm.get_index_futures_contracts(key=key)
    if contract:
        print(f'  {key}: current={contract.current_contract.code if contract.current_contract else "?"}, '
              f'next={contract.next_contract.code if contract.next_contract else "?"}')
    else:
        print(f'  {key}: 无数据')
print()

# ---- 到期预警 ----
print('=== 到期预警测试 ===')
warnings = cm.check_expiry_warnings()
print(f'  到期预警数: {len(warnings)}')
for w in warnings[:3]:
    print(f'    {w}')

print('\nContractManager 测试完成!')

### 3.2 OptionCodeParser — 三格式期权代码解析

In [ ]:
# ============================================================
# 3.2 OptionCodeParser — 三格式期权代码解析
# ============================================================
from subsystems.market_state.core.option_code_parser import OptionCodeParser

parser = OptionCodeParser()

# 格式1: 上交所ETF期权 (510050C6A02850)
print('=== 格式1: 上交所ETF期权 ===')
r1 = parser.parse('510050C6A02850')
if r1:
    print(f'  代码: 510050C6A02850')
    print(f'  标的: {r1.underlying_code}, 类型: {r1.option_type}, 行权价: {r1.strike_price}')
    print(f'  到期年月: {r1.expiry_year}/{r1.expiry_month}')
else:
    print('  解析失败')
print()

# 格式2: 深交所ETF期权 (159901C6M003100A)
print('=== 格式2: 深交所ETF期权 ===')
r2 = parser.parse('159901C6M003100A')
if r2:
    print(f'  代码: 159901C6M003100A')
    print(f'  标的: {r2.underlying_code}, 类型: {r2.option_type}, 行权价: {r2.strike_price}')
else:
    print('  解析失败')
print()

# 格式3: 中金所股指期权 (HO2606-C-2400)
print('=== 格式3: 中金所股指期权 ===')
r3 = parser.parse('HO2606-C-2400')
if r3:
    print(f'  代码: HO2606-C-2400')
    print(f'  标的: {r3.underlying_code}, 类型: {r3.option_type}, 行权价: {r3.strike_price}')
    print(f'  品种: {r3.variety}, 到期月份: {r3.expiry_year}/{r3.expiry_month}')
else:
    print('  解析失败')
print()

# 批量解析测试
print('=== 批量解析测试 ===')
test_codes = ['510050C6A02850', '510300P6A04700', 'IO2606-C-3800', 'MO2606-P-7000']
results = parser.parse_batch(test_codes)
for code, info in results.items():
    if info:
        print(f'  {code}: {info.option_type} {info.underlying_code} K={info.strike_price}')
    else:
        print(f'  {code}: 解析失败')

print('\nOptionCodeParser 测试完成!')

### 3.3 DerivativesSignalEngine — 衍生品信号计算

In [ ]:
# ============================================================
# 3.3 DerivativesSignalEngine — 衍生品信号引擎
# ============================================================
from subsystems.market_state.core.derivatives_signal_engine import DerivativesSignalEngine

signal_engine = DerivativesSignalEngine(
    data_service=data_loader,
    contract_manager=cm,
    overseas_signal_engine=None,
    config=config,
    cache=cache,
)

print('DerivativesSignalEngine 初始化完成')
print(f'  商品品种数: {len(signal_engine._commodity_varieties)}')
print(f'  股指期货数: {len(signal_engine._index_futures)}')
print()

# ---- 计算全量信号 ----
print('=== 全量信号计算 ===')
try:
    result = signal_engine.calculate_all()
    if result:
        print(f'  综合信号: {result.composite_signal:.1f}')
        print(f'  综合方向: {result.composite_direction}')
        print(f'  商品信号数: {len(result.commodity_signals) if result.commodity_signals else 0}')
        print(f'  期限结构信号: {result.term_structure_signal}')
        print(f'  股指基差信号: {result.index_futures_basis}')
    else:
        print('  计算结果为空')
except Exception as e:
    print(f'  计算异常: {e}')

print('\nDerivativesSignalEngine 测试完成!')

### 3.4 PlotlyVisualizer — 交互可视化

In [ ]:
# ============================================================
# 3.4 PlotlyVisualizer — Plotly 6.7.0 交互可视化
# ============================================================
from subsystems.market_state.visualization.plotly_visualizer import PlotlyVisualizer

viz = PlotlyVisualizer(config=config)
print('PlotlyVisualizer 初始化完成')

# 使用已有数据生成测试图表
try:
    # 构造模拟数据用于可视化测试
    mock_result = {
        'classification': {
            'composite_score': 62.5,
            'state_label': '积极配置',
            'direction': 'bullish',
            'scores': {
                'valuation': {'score': 70, 'label': '偏低'},
                'momentum': {'score': 55, 'label': '中性偏多'},
                'regime': {'score': 65, 'label': '温和'},
                'overseas': {'score': 60, 'label': '中性'},
            },
        },
        'derivatives': {
            'composite_signal': 58.3,
            'composite_direction': 'normal',
        },
        'pcr': {
            'composite_pcr': {'volume_pcr': 0.85, 'oi_pcr': 0.92},
        },
        'risk': {
            'overall_risk_score': 38.5,
            'risk_level': 'moderate',
        },
        'regime': {
            'current_regime': 'volatile',
            'regime_label': '震荡',
        },
    }

    # 生成4D雷达图
    fig_radar = viz.plot_market_state_4d(mock_result.get('classification', {}))
    if fig_radar:
        print(f'4D雷达图生成成功')

    # 保存HTML
    output_dir = PROJECT_ROOT / 'output'
    output_dir.mkdir(exist_ok=True)
    figures = viz.generate_all(mock_result, save_html=True, save_json=False)
    print(f'生成的图表数: {len(figures)}')
    for name in figures.keys():
        print(f'  - {name}')

except Exception as e:
    print(f'可视化测试异常: {e}')

print('\nPlotlyVisualizer 测试完成!')

---
## 模块4: 端到端管线测试

### 4.1 MarketStateEngine 子系统完整测试

In [ ]:
# ============================================================
# 4.1 MarketStateEngine — 子系统端到端测试
# ============================================================
from subsystems.market_state.core.market_state_engine import MarketStateEngine

# 注册所有依赖到容器
container.register_singleton('tdx_adapter', tdx)
container.register_singleton('ak_adapter', ak)
container.register_singleton('db_reader', db)
container.register_singleton('data_loader', data_loader)

# 创建 MarketStateEngine
engine = MarketStateEngine(container)
print(f'MarketStateEngine 创建完成, name={engine.name}')
print()

# 启动
print('=== 启动 MarketStateEngine ===')
engine.start()
print(f'  is_running: {engine.is_running}')
print(f'  ContractManager: {"已初始化" if engine.contract_manager else "未初始化"}')
print(f'  SignalEngine: {"已初始化" if engine.signal_engine else "未初始化"}')
print()

# 运行
print('=== 运行 MarketStateEngine ===')
try:
    result = engine.run()
    if result:
        print(f'  综合信号: {result.composite_signal:.1f}')
        print(f'  综合方向: {result.composite_direction}')
        print(f'  计算成功!')
    else:
        print('  计算结果为空')
except Exception as e:
    print(f'  运行异常: {e}')
print()

# 生成报告
print('=== 生成市场状态报告 ===')
try:
    report = engine.generate_report()
    print(f'  报告键数: {len(report)}')
    for k in report.keys():
        print(f'    - {k}')
except Exception as e:
    print(f'  报告生成异常: {e}')
print()

# 停止
engine.stop()
print(f'  is_running: {engine.is_running}')

print('\nMarketStateEngine 端到端测试完成!')

### 4.2 完整 Bootstrap 管线测试

In [ ]:
# ============================================================
# 4.2 完整 Bootstrap + run_pipeline 端到端测试
# ============================================================
from main import bootstrap, run_pipeline, _shutdown

print('=== AiStock V10 完整管线测试 ===')
print()

try:
    # 引导启动
    container = bootstrap(config_dir=str(PROJECT_ROOT / 'config' / 'yaml'))
    print(f'Bootstrap 完成, 已注册服务: {container.list_services()}')
    print()
    
    # 运行管线
    result = run_pipeline(container)
    print(f'管线运行完成')
    print(f'  结果键: {list(result.keys())}')
    for key, val in result.items():
        if isinstance(val, dict):
            print(f'    {key}: {len(val)} items')
        else:
            print(f'    {key}: {type(val).__name__}')
    print()
    
    # 关闭
    _shutdown(container)
    print('系统已关闭')
    
except Exception as e:
    print(f'管线异常: {e}')
    import traceback
    traceback.print_exc()

print('\nAiStock V10 完整管线测试完成!')

### 4.3 配置热重载测试

In [ ]:
# ============================================================
# 4.3 配置热重载测试
# ============================================================
from base_service.config_service import ConfigService
from base_service.event_bus import EventBus

# 创建支持热重载的 ConfigService
event_bus = EventBus()
config_hot = ConfigService(
    config_dir=str(PROJECT_ROOT / 'config' / 'yaml'),
    enable_hot_reload=True,
    hot_reload_interval=2.0,  # 2秒检查一次
    event_bus=event_bus,
)
config_hot.load_all()

print(f'ConfigService 热重载已启动')
print(f'  检查间隔: 2.0s')
print(f'  已加载 YAML 数: {len(config_hot._configs)}')

# 订阅配置变更事件
change_events = []
def on_config_change(event):
    change_events.append(event)
    print(f'  配置变更事件: {event.data}')

event_bus.subscribe('config.changed', on_config_change)
print(f'  已订阅 config.changed 事件')

# 验证配置内容
futures_before = config_hot.get_futures_codes()
print(f'  期货代码数(当前): {len(futures_before)}')

# 测试手动重载
import time
print('\n手动重载测试...')
config_hot.reload()
futures_after = config_hot.get_futures_codes()
print(f'  期货代码数(重载后): {len(futures_after)}')
print(f'  配置一致性: {len(futures_before) == len(futures_after)}')

# 停止热重载
config_hot.stop()
print('\n热重载已停止')

print('配置热重载测试完成!')

### 4.4 代码表对齐验证测试

In [ ]:
# ============================================================
# 4.4 tdxAPICode180.xlsx 代码表对齐验证
# ============================================================
import pandas as pd

xlsx_path = str(PROJECT_ROOT / 'data' / 'tdxAPICode180.xlsx')
if os.path.exists(xlsx_path):
    df = pd.read_excel(xlsx_path)
    print(f'tdxAPICode180.xlsx 已加载')
    print(f'  总行数: {len(df)}')
    print(f'  列名: {list(df.columns)[:8]}')
    print()
    
    # 验证V10期货代码与xlsx对齐
    if 'code' in df.columns:
        xlsx_codes = set(df['code'].astype(str).tolist())
        futures_from_yaml = config.get_futures_codes()
        
        print('=== 期货代码对齐验证 ===')
        for fc in futures_from_yaml:
            code = fc.get('code', '')
            name = fc.get('name', '')
            in_xlsx = code in xlsx_codes
            status = 'OK' if in_xlsx else 'MISMATCH!'
            print(f'  {code} ({name}): {status}')
        print()
        
        # 验证指数代码
        indices_from_yaml = config.get_index_codes()
        print('=== 指数代码对齐验证 ===')
        for ic in indices_from_yaml:
            code = ic.get('code', '')
            name = ic.get('name', '')
            # 指数代码可能不直接在xlsx中
            print(f'  {code} ({name}): 配置已定义')
        print()
        
        # 验证L8/L9后缀
        print('=== L8/L9连续合约后缀验证 ===')
        l8_codes = [c for c in xlsx_codes if c.endswith('L8')]
        l9_codes = [c for c in xlsx_codes if c.endswith('L9')]
        m0_codes = [c for c in xlsx_codes if c.endswith('M0')]
        print(f'  L8(主连)代码数: {len(l8_codes)}')
        print(f'  L9(加权)代码数: {len(l9_codes)}')
        print(f'  M0(不存在!)代码数: {len(m0_codes)}')
        if l8_codes:
            print(f'  L8示例: {l8_codes[:5]}')
        if m0_codes:
            print(f'  警告: 发现M0代码, TDX不支持M0后缀!')
        else:
            print(f'  确认: 无M0代码 (正确)')
else:
    print(f'代码表文件不存在: {xlsx_path}')

print('\n代码表对齐验证测试完成!')

---
## V10 架构升级总结

| 特性 | V9 | V10 |
|------|-----|------|
| 配置方式 | Python硬编码 + 单YAML | 8个独立YAML + 4层合并 |
| 配置变更 | 需全代码审计 | 改YAML一处全局生效 |
| 热重载 | 无 | ConfigService mtime轮询 + EventBus通知 |
| 子系统隔离 | 无子系统概念 | SubsystemBase + 命名空间缓存 |
| 子系统通信 | 无 | EventBus topic pub/sub + 通配符 |
| 服务管理 | 手动初始化 | ServiceContainer DI容器 |
| 缓存 | 单一CacheService | 命名空间隔离 + 按段TTL |
| 期货后缀 | L8(已修正) | L8(从YAML加载) |
| 代码表 | tdxAPICode180.xlsx | 同 + ConfigService驱动 |
| 未来扩展 | 需改代码 | 新增子系统+YAML即可 |